# Fine-Tuning Phi-2 for XSum Text Summarization

**Repository:** `finetuning-phi-2-text-summarization`  
**Task:** UAS Deep Learning — Fine-Tuning HuggingFace Models, Task 3  
**Architecture Type:** Decoder-only Transformer / Causal Language Model  
**Model:** `microsoft/phi-2`  
**Dataset:** XSum  
**Problem Type:** Abstractive Text Summarization

---

## 1. Learning Objectives

By the end of this notebook, you should be able to:

1. Build an end-to-end decoder-only LLM fine-tuning pipeline using HuggingFace.
2. Load and inspect the XSum summarization dataset.
3. Convert document-summary pairs into instruction-style causal language modeling prompts.
4. Fine-tune Phi-2 using parameter-efficient LoRA configuration.
5. Generate abstractive summaries from unseen documents.
6. Evaluate summary quality using ROUGE-style metrics.
7. Analyze generated summaries qualitatively and quantitatively.

## 2. Theoretical Background

Phi-2 is a decoder-only Transformer language model. Unlike encoder-decoder models such as T5, a decoder-only model predicts the next token based on previous tokens. To adapt a decoder-only model for summarization, the input is written as an instruction prompt containing the document and a `Summary:` marker. The model then learns to continue the text by generating the target summary.

XSum is an abstractive summarization dataset. The summaries are usually concise and require rewriting, not only sentence extraction. This makes XSum suitable for testing whether a model can compress and rewrite information.

Because Phi-2 is larger than DistilBERT and T5-base, this notebook uses LoRA for parameter-efficient fine-tuning. LoRA trains a small number of adapter parameters instead of updating all model weights, making the experiment more practical on Google Colab GPU.

## 3. Environment Setup

Run this notebook in Google Colab with **T4 GPU** enabled. If required libraries are missing, uncomment and run the installation command.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U transformers datasets accelerate peft bitsandbytes scikit-learn matplotlib pandas numpy

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.9 MB/s eta 0:00:00
ERROR: p

## 4. Import Libraries and Set Configuration

This configuration is intentionally lightweight. Phi-2 is larger than the previous models, so the notebook uses a small subset, short sequence length, LoRA, and 4-bit loading when available.

In [ ]:
import os
import re
import math
import random
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

RANDOM_STATE = 42
MODEL_NAME = "microsoft/phi-2"
DATASET_NAME = "EdinburghNLP/xsum"
OUTPUT_DIR = "./phi2-xsum-checkpoints"
FINAL_MODEL_DIR = "./phi2-xsum-lora-final"

TRAIN_SIZE = 300
VAL_SIZE = 50
TEST_SIZE = 50
MAX_LENGTH = 512
MAX_NEW_TOKENS = 64

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Model: microsoft/phi-2
Dataset: EdinburghNLP/xsum
CUDA available: True
GPU: Tesla T4


## 5. Load XSum Dataset

The assignment specifies XSum for Task 3. XSum contains news documents and short abstractive summaries.

In [ ]:
try:
    dataset = load_dataset(DATASET_NAME)
except Exception as e:
    print("Primary dataset failed to load:", e)
    print("Falling back to 'xsum'.")
    dataset = load_dataset("xsum")

print(dataset)
print("Available splits:", dataset.keys())
print("Training example keys:", dataset["train"][0].keys())
print("Example:")
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})
Available splits: dict_keys(['train', 'validation', 'test'])
Training example keys: dict_keys(['document', 'summary', 'id'])
Example:
{'document': 'The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.\nRepair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.\nTrains on the west coast mainline face disruption due to damage at the Lamington Viaduct.\nMany businesses and householders were affected by flooding in Newton Stewart after the River Cree overflowed into the town.\nFirst Minister Nicola Sturgeon visited the area to inspect the damage.\nThe waters breached a retaining

## 6. Dataset Inspection

Before fine-tuning, inspect the document and summary columns. The `document` field is the input article, and the `summary` field is the target abstractive summary.

In [ ]:
preview_df = pd.DataFrame(dataset["train"].select(range(5)))
print(preview_df[["id", "document", "summary"]])

for idx in range(2):
    row = dataset["train"][idx]
    print("\nExample", idx)
    print("Document:", row["document"][:500], "...")
    print("Summary:", row["summary"])

         id                                           document  \
0  35232142  The full cost of damage in Newton Stewart, one...   
1  40143035  A fire alarm went off at the Holiday Inn in Ho...   
2  35951548  Ferrari appeared in a position to challenge un...   
3  36266422  John Edward Bates, formerly of Spalding, Linco...   
4  38826984  Patients and staff were evacuated from Cerahpa...   

                                             summary  
0  Clean-up operations are continuing across the ...  
1  Two tourist buses have been destroyed by fire ...  
2  Lewis Hamilton stormed to pole position at the...  
3  A former Lincolnshire Police officer carried o...  
4  An armed man who locked himself into a room at...  

Example 0
Document: The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.
Repair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.
Trains on the west coast mainline face disrupt

## 7. Create Lightweight Training, Validation, and Test Subsets

The full XSum dataset is large. This notebook uses a small subset to keep the experiment practical on Google Colab. The goal is to demonstrate a complete pipeline, not to maximize leaderboard performance.

In [ ]:
train_split = dataset["train"].shuffle(seed=RANDOM_STATE)
val_split = dataset["validation"].shuffle(seed=RANDOM_STATE)

tiny_train = train_split.select(range(min(TRAIN_SIZE, len(train_split))))
tiny_val = val_split.select(range(min(VAL_SIZE, len(val_split))))
tiny_test = val_split.select(range(min(VAL_SIZE, len(val_split)), min(VAL_SIZE + TEST_SIZE, len(val_split))))

print("Train size:", len(tiny_train))
print("Validation size:", len(tiny_val))
print("Test size:", len(tiny_test))

Train size: 300
Validation size: 50
Test size: 50


## 8. Format XSum for Decoder-Only Instruction Fine-Tuning

Decoder-only models are trained to continue text. The notebook builds an instruction prompt and appends the reference summary as the continuation target.

In [ ]:
def build_prompt(document):
    return (
        "Summarize the following news article in one concise sentence.\n\n"
        f"Document:\n{document.strip()}\n\n"
        "Summary:"
    )


def format_example(example):
    prompt = build_prompt(example["document"])
    summary = example["summary"].strip()
    full_text = prompt + " " + summary
    return {
        "prompt": prompt,
        "target_summary": summary,
        "full_text": full_text
    }

formatted_train = tiny_train.map(format_example)
formatted_val = tiny_val.map(format_example)
formatted_test = tiny_test.map(format_example)

print("Prompt example:")
print(formatted_train[0]["prompt"][:800])
print("\nTarget summary:")
print(formatted_train[0]["target_summary"])

Prompt example:
Summarize the following news article in one concise sentence.

Document:
In Wales, councils are responsible for funding and overseeing schools.
But in England, Mr Osborne's plan will mean local authorities will cease to have a role in providing education.
Academies are directly funded by central government and head teachers have more freedom over admissions and to change the way the school works.
It is a significant development in the continued divergence of schools systems on either side of Offa's Dyke.
And although the Welsh Government will get extra cash to match the money for English schools to extend the school day, it can spend it on any devolved policy area.
Ministers have no plans to follow suit.
At the moment, governing bodies are responsible for setting school hours and they need

Target summary:
As Chancellor George Osborne announced all English state schools will become academies, the Welsh Government continues to reject the model here.


## 9. Load Tokenizer and Tokenize Dataset

The tokenizer converts instruction-style training examples into token IDs. The tokenizer pad token is set to the EOS token if no pad token exists.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(batch):
    return tokenizer(
        batch["full_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

tokenized_train = formatted_train.map(tokenize_function, batched=True)
tokenized_val = formatted_val.map(tokenize_function, batched=True)
tokenized_test = formatted_test.map(tokenize_function, batched=True)

print(tokenized_train[0].keys())
print("Tokenized length:", len(tokenized_train[0]["input_ids"]))

dict_keys(['document', 'summary', 'id', 'prompt', 'target_summary', 'full_text', 'input_ids', 'attention_mask'])
Tokenized length: 512


## 10. Load Phi-2 with 4-bit Quantization and LoRA

This cell loads Phi-2 using 4-bit quantization when supported by the runtime. LoRA adapters are then attached so that only a small number of trainable parameters are updated.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    print("Loaded Phi-2 with 4-bit quantization.")
except Exception as e:
    print("4-bit loading failed:", e)
    print("Falling back to float16 loading. This may require more GPU memory.")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )

model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "dense"]
)

try:
    model = get_peft_model(model, lora_config)
except Exception as e:
    print("Default target_modules failed:", e)
    print("Retrying with a safer module list for Phi-style architectures.")
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj"]
    )
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loaded Phi-2 with 4-bit quantization.
trainable params: 5,242,880 || all params: 2,784,926,720 || trainable%: 0.1883


## 11. Training Configuration

The training setup is intentionally small. Gradient accumulation is used to simulate a larger batch size while keeping GPU memory usage low.

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=25,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",
    fp16=True,
    seed=RANDOM_STATE
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

print("Trainer configured.")

Trainer configured.


## 12. Fine-Tune the Model

This cell fine-tunes the LoRA adapters on the XSum subset. If the runtime runs out of memory, reduce `TRAIN_SIZE`, reduce `MAX_LENGTH`, or restart with a fresh T4 GPU runtime.

In [ ]:
train_result = trainer.train()
print(train_result)

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,2.470495,2.287577


TrainOutput(global_step=38, training_loss=2.4499791295904862, metrics={'train_runtime': 274.5579, 'train_samples_per_second': 1.093, 'train_steps_per_second': 0.138, 'total_flos': 1825491123118080.0, 'train_loss': 2.4499791295904862, 'epoch': 1.0})


## 13. Generate Summaries on the Test Subset

After fine-tuning, generate summaries from held-out XSum documents.

In [ ]:
def generate_summary(document):
    prompt = build_prompt(document)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=2,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Summary:" in decoded:
        return decoded.split("Summary:")[-1].strip()
    return decoded.strip()

pred_summaries = []
ref_summaries = []

for example in tiny_test:
    pred = generate_summary(example["document"])
    ref = example["summary"].strip()
    pred_summaries.append(pred)
    ref_summaries.append(ref)

print("Generated", len(pred_summaries), "summaries.")
for i in range(min(3, len(pred_summaries))):
    print("\nDocument:", tiny_test[i]["document"][:300], "...")
    print("Prediction:", pred_summaries[i])
    print("Reference:", ref_summaries[i])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Generated 50 summaries.

Document: City led 2-0 at half-time but lost 3-2 after Kieran Lee's 96th-minute winner.
Both teams had a player sent off, with Gary O'Neil dismissed seconds after Lee Tomlin's penalty bounced clear off the post when Johnson's side were 2-1 up.
"There was a turning point in the game. You couldn't make it up, i ...
Prediction: City lost 3-2 at home to Bristol City after a late winner from Kieran Lee.

Possible summary:

City suffered a 3-2 home defeat to Bristol City after Lee Tomlin scored a late winner.
Reference: Bristol City head coach Lee Johnson says their loss at Sheffield Wednesday showed him which of his players "can't quite handle the pressure."

Document: Milford Haven Coastguard were alerted to the incident near Blaenplwyf just before 14:40 BST.
A Sea King rescue helicopter from RAF Valley flew the man to Morriston Hospital in Swansea. He has been described as dazed but conscious.
A second man was also rescued from the cliff face but was reported to .

## 14. ROUGE-Style Evaluation

ROUGE measures overlap between generated and reference summaries. This notebook implements simple ROUGE-1, ROUGE-2, and ROUGE-L F1 scores without requiring an external evaluation package.

In [ ]:
def tokenize_for_rouge(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()


def ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)])) if len(tokens) >= n else []


def rouge_n_f1(prediction, reference, n=1):
    pred_tokens = tokenize_for_rouge(prediction)
    ref_tokens = tokenize_for_rouge(reference)
    pred_ngrams = collections.Counter(ngrams(pred_tokens, n))
    ref_ngrams = collections.Counter(ngrams(ref_tokens, n))
    overlap = sum((pred_ngrams & ref_ngrams).values())
    if overlap == 0:
        return 0.0
    precision = overlap / max(sum(pred_ngrams.values()), 1)
    recall = overlap / max(sum(ref_ngrams.values()), 1)
    return 2 * precision * recall / (precision + recall)


def lcs_length(x, y):
    dp = [[0] * (len(y) + 1) for _ in range(len(x) + 1)]
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            if x[i - 1] == y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]


def rouge_l_f1(prediction, reference):
    pred_tokens = tokenize_for_rouge(prediction)
    ref_tokens = tokenize_for_rouge(reference)
    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0
    lcs = lcs_length(pred_tokens, ref_tokens)
    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

rouge1_scores = [rouge_n_f1(p, r, n=1) for p, r in zip(pred_summaries, ref_summaries)]
rouge2_scores = [rouge_n_f1(p, r, n=2) for p, r in zip(pred_summaries, ref_summaries)]
rougeL_scores = [rouge_l_f1(p, r) for p, r in zip(pred_summaries, ref_summaries)]

rouge_metrics = {
    "rouge1": float(np.mean(rouge1_scores)),
    "rouge2": float(np.mean(rouge2_scores)),
    "rougeL": float(np.mean(rougeL_scores))
}
print("ROUGE-style metrics:")
print(rouge_metrics)

ROUGE-style metrics:
{'rouge1': 0.17271614260798507, 'rouge2': 0.05096470336118588, 'rougeL': 0.13017800030173907}


## 15. Inference Demo

This section tests the fine-tuned model on a custom short news paragraph.

In [ ]:
custom_document = (
    "A new satellite was launched to monitor climate patterns and improve weather forecasting. "
    "Scientists said the mission will help collect better data about storms, rainfall, and temperature changes."
)
custom_summary = generate_summary(custom_document)
print("Custom document:")
print(custom_document)
print("\nGenerated summary:")
print(custom_summary)

Custom document:
A new satellite was launched to monitor climate patterns and improve weather forecasting. Scientists said the mission will help collect better data about storms, rainfall, and temperature changes.

Generated summary:
A new satellite was launched to monitor climate patterns and improve weather forecasting.


## 16. Final Conclusion

This notebook demonstrates an end-to-end HuggingFace fine-tuning pipeline for abstractive text summarization using Phi-2 and the XSum dataset. The pipeline includes dataset loading, instruction-style prompt formatting, tokenizer preparation, LoRA-based fine-tuning, summary generation, ROUGE-style evaluation, and inference testing.

After the notebook is executed, this conclusion should be updated using the final ROUGE-1, ROUGE-2, and ROUGE-L scores from the evaluation section.